# Chapter 5 — Learning Multiple Weights at a Time: Generalizing Gradient Descent

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 5:** Learning Multiple Weights at a Time: Generalizing Gradient Descent

## Isi Chapter 5

Chapter sebelumnya memperbarui satu weight. Chapter ini memperluas gradient descent
ke banyak weight, dalam beberapa kasus:

1. Banyak input, satu output
2. Membekukan (freeze) satu weight
3. Banyak output
4. Banyak input dan banyak output (weights berbentuk matriks)
5. Bagaimana weight belajar, termasuk contoh MNIST
6. Dot product sebagai ukuran kemiripan

## 1. Gradient descent dengan banyak input

Sekarang ada tiga weight dan tiga input (`toes`, `wlrec`, `nfans`). Langkahnya tetap
predict, compare, learn. Yang berubah adalah tahap learn: setiap weight memiliki
`weight_delta` sendiri, yaitu `delta` dikali input yang terhubung dengannya.

In [ ]:
def w_sum(a, b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

def neural_network(input, weights):
    pred = w_sum(input, weights)
    return pred

def ele_mul(number, vector):
    output = [0, 0, 0]
    assert(len(output) == len(vector))
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

toes  = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

win_or_lose_binary = [1, 1, 0, 1]
true = win_or_lose_binary[0]

weights = [0.1, 0.2, -0.1]
input = [toes[0], wlrec[0], nfans[0]]
alpha = 0.01

# 1. Predict
pred = neural_network(input, weights)
# 2. Compare
error = (pred - true) ** 2
delta = pred - true
# 3. Learn
weight_deltas = ele_mul(delta, input)
for i in range(len(weights)):
    weights[i] -= alpha * weight_deltas[i]

print('pred          :', round(pred, 3))
print('error         :', round(error, 4))
print('delta         :', round(delta, 3))
print('weight_deltas :', [round(w, 3) for w in weight_deltas])
print('weights baru  :', [round(w, 4) for w in weights])

Pada satu iterasi, prediksi `0.86` dan `delta` `-0.14`. `weight_deltas` menjadi
`[-1.19, -0.091, -0.168]`. Setelah dikali `alpha = 0.01`, weights berubah menjadi
`[0.1119, 0.2009, -0.0983]`.

Saya menjalankan tiga iterasi untuk melihat perubahan prediksi dan error:

In [ ]:
weights = [0.1, 0.2, -0.1]
alpha = 0.01

for it in range(3):
    pred = neural_network(input, weights)
    error = (pred - true) ** 2
    delta = pred - true
    weight_deltas = ele_mul(delta, input)
    for i in range(len(weights)):
        weights[i] -= alpha * weight_deltas[i]
    print('Iterasi', it + 1,
          '| pred', round(pred, 4),
          '| error', round(error, 4),
          '| weights', [round(w, 4) for w in weights])

Error turun dari `0.0196` ke `0.0013` lalu `0.0001`. Ketiga weight bergerak bersamaan
ke arah yang mengurangi error, tetapi besarnya berbeda karena `weight_delta`-nya
berbeda.

## 2. Membekukan satu weight

Saya membekukan weight pertama dengan `weight_deltas[0] = 0`, dan memakai
`alpha = 0.3`. Weight pertama tidak pernah berubah, sedangkan dua weight lainnya tetap
dilatih.

In [ ]:
weights = [0.1, 0.2, -0.1]
alpha = 0.3

for it in range(3):
    pred = neural_network(input, weights)
    error = (pred - true) ** 2
    delta = pred - true
    weight_deltas = ele_mul(delta, input)
    weight_deltas[0] = 0          # weight pertama dibekukan
    for i in range(len(weights)):
        weights[i] -= alpha * weight_deltas[i]
    print('Iterasi', it + 1,
          '| pred', round(pred, 4),
          '| error', round(error, 6),
          '| weights', [round(w, 4) for w in weights])

Weight pertama tetap `0.1`, tetapi error tetap mengecil. Dua weight lainnya menyesuaikan
diri untuk menutup selisih itu. Jika dilanjutkan sampai 10 iterasi, prediksi mendekati
`1.0` dan error sekitar `1e-08`, walaupun weight pertama tidak pernah berubah.

Pelajaran yang saya tangkap: jaringan tidak perlu memakai semua input untuk menurunkan
error. Weight yang dibekukan tidak ikut belajar, tetapi weight lain tetap bisa mencapai
hasil yang benar.

## 3. Gradient descent dengan banyak output

Pada bagian ini satu input (`wlrec`) menghasilkan tiga output: `hurt`, `win`, dan `sad`.
Setiap output memiliki `delta` sendiri, dan setiap weight dihitung dari input dikali
delta output yang terhubung dengannya.

In [ ]:
def scalar_ele_mul(number, vector):
    output = [0, 0, 0]
    assert(len(output) == len(vector))
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

def neural_network_multi(input, weights):
    return [input * w for w in weights]   # satu input, tiga output

weights = [0.3, 0.2, 0.9]

wlrec = [0.65, 1.0, 1.0, 0.9]
hurt  = [0.1, 0.0, 0.0, 0.1]
win   = [1,   1,   0,   1]
sad   = [0.1, 0.0, 0.1, 0.2]

input = wlrec[0]
true = [hurt[0], win[0], sad[0]]
alpha = 0.1

for it in range(3):
    pred = neural_network_multi(input, weights)
    error = [0, 0, 0]
    delta = [0, 0, 0]
    for i in range(len(true)):
        error[i] = (pred[i] - true[i]) ** 2
        delta[i] = pred[i] - true[i]
    weight_deltas = scalar_ele_mul(input, delta)
    for i in range(len(weights)):
        weights[i] -= alpha * weight_deltas[i]
    print('Iterasi', it + 1,
          '| pred', [round(p, 3) for p in pred],
          '| error', [round(e, 3) for e in error])

Pada iterasi pertama, prediksi adalah `[0.195, 0.13, 0.585]`. Pada iterasi berikutnya,
prediksi `hurt` dan `win` bergerak ke arah target, sedangkan prediksi `sad` juga turun.
Setiap output hanya bergantung pada weight-nya sendiri dan delta-nya sendiri, sehingga
koreksinya tidak saling mengganggu.

## 4. Gradient descent dengan banyak input dan output

Ketika ada banyak input dan banyak output, weights berbentuk matriks. Setiap
`weight_deltas[i][j]` dihitung dari input ke-`i` dikali delta output ke-`j`. Operasi
ini disebut **outer product**.

In [ ]:
def w_sum(a, b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

def vect_mat_mul(vect, matrix):
    assert(len(vect) == len(matrix))
    output = [0, 0, 0]
    for i in range(len(vect)):
        output[i] = w_sum(vect, matrix[i])
    return output

def neural_network(input, weights):
    return vect_mat_mul(input, weights)

def zeros_matrix(rows, cols):
    return [[0] * cols for _ in range(rows)]

def outer_prod(vec_a, vec_b):
    out = zeros_matrix(len(vec_a), len(vec_b))
    for i in range(len(vec_a)):
        for j in range(len(vec_b)):
            out[i][j] = vec_a[i] * vec_b[j]
    return out

toes  = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

hurt = [0.1, 0.0, 0.0, 0.1]
win  = [1,   1,   0,   1]
sad  = [0.1, 0.0, 0.1, 0.2]

weights = [[0.1, 0.1, -0.3],   # hurt?
           [0.1, 0.2,  0.0],   # win?
           [0.0, 1.3,  0.1]]   # sad?

alpha = 0.01
input = [toes[0], wlrec[0], nfans[0]]
true = [hurt[0], win[0], sad[0]]

for it in range(3):
    pred = neural_network(input, weights)
    error = [0, 0, 0]
    delta = [0, 0, 0]
    for i in range(len(true)):
        error[i] = (pred[i] - true[i]) ** 2
        delta[i] = pred[i] - true[i]
    weight_deltas = outer_prod(input, delta)
    for i in range(len(weights)):
        for j in range(len(weights[0])):
            weights[i][j] -= alpha * weight_deltas[i][j]
    print('Iterasi', it + 1,
          '| pred', [round(p, 4) for p in pred],
          '| error', [round(e, 4) for e in error])

`weight_deltas` di sini berbentuk matriks 3x3 dengan ukuran yang sama dengan `weights`,
sehingga setiap weight mendapat koreksinya sendiri dari input dan delta yang
terhubung dengannya.

Hasilnya tidak selalu mulus. Pada iterasi pertama, prediksi `hurt` melompat dari
`0.555` ke `0.139`, karena input `toes` bernilai besar (`8.5`) dan `alpha = 0.01`
masih cukup besar untuk input itu. Output `sad` turun terus, tetapi error `hurt`
dan `win` naik sedikit pada iterasi ketiga. Saya menilai ini sebagai tanda bahwa
gradient descent bisa melewati titik terbaik ketika langkahnya terlalu besar,
sehingga `alpha` perlu dipilih dengan hati-hati.

## 5. Bagaimana weight belajar (MNIST)

Dataset **MNIST** berisi gambar digit tulisan tangan 28x28 piksel. Setiap gambar
diratakan menjadi 784 input, dan jaringan memiliki 10 output (satu untuk setiap digit).
Arsitekturnya sama dengan bagian 4, hanya jumlah input dan outputnya yang berbeda.

Setiap output memiliki 784 weight, satu untuk setiap piksel. Weight yang positif berarti
piksel tersebut mendukung digit itu, dan weight yang negatif berarti piksel tersebut
menentang digit itu. Jika weights untuk satu digit divisualisasikan sebagai gambar,
bentuk kasar digit tersebut akan terlihat. Dataset MNIST tidak tersedia di sandbox ini,
jadi bagian ini hanya menjelaskan konsepnya.

## 6. Dot product sebagai ukuran kemiripan

Dot product dua vektor menghasilkan skor tinggi jika keduanya bernilai positif pada
posisi yang sama, dan menghasilkan 0 jika tidak ada posisi yang bertumpang tindih.

In [ ]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

a = [0, 1, 0, 1]
b = [1, 0, 1, 0]
c = [0, 1, 1, 0]
d = [0.5, 0, 0.5, 0]

print('dot(a, b) =', dot(a, b))   # tidak ada posisi yang sama -> 0
print('dot(b, c) =', dot(b, c))   # satu posisi sama -> 1
print('dot(c, d) =', dot(c, d))   # posisi sama dengan nilai 0.5 -> 0.5
print('dot(c, c) =', dot(c, c))   # vektor identik -> skor tertinggi

## Key Ideas — Chapter 5

- Gradient descent berlaku untuk banyak weight, dan setiap weight punya `weight_delta` sendiri.
- Untuk banyak input dan satu output, `weight_delta` setiap input adalah `input * delta`.
- Untuk banyak output, setiap output punya delta sendiri, dan weight dihitung dari input dikali delta output tersebut.
- Untuk banyak input dan output, `weight_delta` adalah **outer product** input dan delta.
- Weight yang dibekukan tidak ikut belajar, tetapi weight lain tetap bisa menurunkan error.
- Dot product adalah ukuran kemiripan dua vektor.

## Kesimpulan

Gradient descent dapat diperluas dari satu weight ke banyak weight tanpa mengubah
prinsipnya: hitung delta di setiap output, kalikan dengan input untuk mendapat
`weight_delta`, lalu perbarui weight dengan `alpha`. Chapter berikutnya menambahkan
lapisan tersembunyi (*hidden layer*) ke jaringan.